## Debug/Clean Run Notebook

### Imports

In [55]:
import os
import pandas as pd
import numpy as np
from relaiss import constants
import relaiss as rl
from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import train_test_split
from pprint import pprint
from sklearn.impute import SimpleImputer
import io, sys, requests
import matplotlib.pyplot as plt

In [57]:
!pip install alerce 
from alerce.core import Alerce
al = Alerce()  # defaults usually work; can pass URLs if needed

### Load Data

In [23]:
csv_path = "/Users/jennakempster-taylor/re-laiss/reference_20kk.csv" 

df = pd.read_csv(csv_path, low_memory=False)

print("Shape:", df.shape)
print("Columns:", df.columns[:458])
print("First few objNames:", df["objName"].head())

Shape: (25515, 458)
Columns: Index(['t0', 'g_peak_mag', 'g_peak_time', 'g_rise_time', 'g_decline_time',
       'g_duration_above_half_flux', 'g_amplitude', 'g_skewness',
       'g_beyond_2sigma', 'r_peak_mag',
       ...
       'zExtNSigma', 'ypsfCore', 'zFmeanflxR6', 'iFmeanflxR6', 'mjd_extracted',
       'tns_redshift', 'latest_magnitude', 'oldest_alert', 'newest_alert',
       'peak_phase'],
      dtype='object', length=458)
First few objNames: 0    PSO J032.5595-01.8136
1    PSO J118.7364+35.9410
2    PSO J158.8836+37.6496
3    PSO J031.2821+11.2486
4    PSO J105.8641+32.0687
Name: objName, dtype: object


In [50]:
print("Rows:", len(df), " Columns:", len(df.columns))
print(df.columns.tolist()[:20])

Rows: 25515  Columns: 458
['t0', 'g_peak_mag', 'g_peak_time', 'g_rise_time', 'g_decline_time', 'g_duration_above_half_flux', 'g_amplitude', 'g_skewness', 'g_beyond_2sigma', 'r_peak_mag', 'r_peak_time', 'r_rise_time', 'r_decline_time', 'r_duration_above_half_flux', 'r_amplitude', 'r_skewness', 'r_beyond_2sigma', 'mean_g-r', 'g-r_at_g_peak', 'mean_color_rate']


Lots of missing alert data. So we apply 200 day cut, but recognise this will not filter out object with missing data

In [52]:
# Overall missingness
na_fraction = df.isna().mean().sort_values(ascending=False)
print(na_fraction.head(20))

# Quick summary stats
print(df.isna().sum().sum(), " total NaNs across all cells.")

host_offset_info               1.000000
objAltName3                    1.000000
host_redshift_info             1.000000
host_2_name                    1.000000
host_name                      1.000000
objAltName1                    1.000000
objAltName2                    1.000000
nStackObjectRows               1.000000
host_2_offset_info             1.000000
host_2_redshift_info           1.000000
tns_redshift                   0.994082
peak_phase                     0.854674
newest_alert                   0.854674
oldest_alert                   0.854674
latest_magnitude               0.854674
g_secondary_peak_width         0.785773
g_dt_main_to_secondary_peak    0.785773
g_secondary_peak_prominence    0.785773
g_dmag_secondary_peak          0.785773
r_dt_main_to_secondary_peak    0.768254
dtype: float64
2171616  total NaNs across all cells.


In [46]:
print(df[["oldest_alert", "newest_alert"]].isna().sum())
print(df[["oldest_alert", "newest_alert"]].notna().sum())

oldest_alert    21807
newest_alert    21807
dtype: int64
oldest_alert    3708
newest_alert    3708
dtype: int64


In [66]:
total = len(df)
present = df["oldest_alert"].notna().sum()
print(f"{present} / {total} = {present/total:.1%} have both alert times")


3708 / 25515 = 14.5% have both alert times


In [70]:
df_valid_alerts = df[df["oldest_alert"].notna()].copy()
df_valid_alerts[["ZTFID", "oldest_alert", "newest_alert"]].head()


,ZTFID,oldest_alert,newest_alert
21807,ZTF18abgsgyj,60744.543831,60798.496620
21808,ZTF24aaipblm,60407.485972,60798.494526
21809,ZTF24aanlwgw,60443.379201,60798.494526
21810,ZTF24aatscgg,60476.397384,60798.494526
21811,ZTF24aakzkae,60428.457454,60798.494526


### Start to cut data

In [135]:
CSV_PATH = "/Users/jennakempster-taylor/re-laiss/reference_20k.csv"   
USE_HOST = False                         

# --- Your feature discovery (as you have it) ---
default_lc_features = constants.lc_features_const.copy()
default_host_features = constants.host_features_const.copy()

def overlap(cols, frame):
    return [c for c in cols if (c in frame.columns) and (not c.endswith("_err"))]

lc_cols   = overlap(default_lc_features, df)
host_cols = overlap(default_host_features, df) if USE_HOST else []
feature_cols = lc_cols + host_cols

print(f"Found {len(lc_cols)} LC features in CSV.")
if USE_HOST:
    print(f"Found {len(host_cols)} host features in CSV.")
print(f"Total features used: {len(feature_cols)}")

### ---- Cut Number 1: Any objects spanning over 200 days -----
# Ensure numeric
df["oldest_alert"] = pd.to_numeric(df["oldest_alert"], errors="coerce")
df["newest_alert"] = pd.to_numeric(df["newest_alert"], errors="coerce")

# Compute span 
span = df["newest_alert"] - df["oldest_alert"]

# Keep rows where:
#   - span is less than 200
#   - OR span is NaN  missing data
mask_span = (span.le(200)) | (span.isna())

# Apply mask
df_cut = df.loc[mask_span].copy()

print(f"Kept {mask_span.sum()} / {len(df)} rows ({mask_span.mean():.1%})")

Found 25 LC features in CSV.
Total features used: 25
Kept 23149 / 25515 rows (90.7%)


In [137]:
# ------ Cut Number 2: Have a minimum lightcurve amplitude ------
# Using df_cut (mask from before)
# Look at g and r amplitudes and 

amp_max = pd.concat([df_cut["g_amplitude"], df_cut["r_amplitude"]], axis=1).max(axis=1, skipna=True)

# Retrieve those with amplitude greater than 0.5 and makes new column amp_flag (yes if more than 0.5)
df_cut["amp_flag"] = amp_max.ge(0.5) 

In [138]:
# ------ Cut Number 3: Use rise time and decline time to cut objects with over 200 days

mask_valid = (
    df_cut["g_rise_time"].between(0, 100, inclusive="neither")
    & df_cut["g_decline_time"].between(0, 200, inclusive="neither")
)

df_valid = df_cut.loc[mask_valid | df_cut["g_rise_time"].isna()].copy() # Add back in NaNs as these may be anomalous

In [107]:
from alerce.core import Alerce
import time
al = Alerce()

# Look at objects with SN like classifications

oids = df["ZTFID"].astype(str).unique().tolist()

# USe cache folder to help with rnu time
from pathlib import Path
CACHE_DIR = Path("./alerce_prob_cache")
CACHE_DIR.mkdir(exist_ok=True, parents=True)
CACHE_FILE = CACHE_DIR / "probabilities.parquet"

# Rerun
if CACHE_FILE.exists():
    probs_all = pd.read_parquet(CACHE_FILE)
    done = set(probs_all["oid"].astype(str))
else:
    probs_all = pd.DataFrame()
    done = set()

to_do = [oid for oid in oids if oid not in done]

# Fetch one object
def fetch_one(oid, retries=3, base_sleep=0.8):
    for a in range(retries):
        try:
            pdf = al.query_probabilities(oid=oid, format="pandas")
            if pdf is not None and len(pdf):
                pdf = pdf.copy()
                pdf["oid"] = oid
                return pdf
            return pd.DataFrame({"oid":[oid]})
        except Exception as e:
            if a == retries - 1:
                return pd.DataFrame({"oid":[oid], "error":[str(e)]})
            time.sleep(base_sleep * (2**a))

from concurrent.futures import ThreadPoolExecutor, as_completed
MAX_WORKERS  = 12
BATCH_FLUSH  = 400

pieces = [probs_all] if len(probs_all) else []
count = 0

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
    futs = {ex.submit(fetch_one, oid): oid for oid in to_do}
    for fut in as_completed(futs):
        res = fut.result()
        if res is not None:
            pieces.append(res)
        count += 1
        if count % BATCH_FLUSH == 0:
            pd.concat(pieces, ignore_index=True).drop_duplicates().to_parquet(CACHE_FILE)

probs_all = pd.concat(pieces, ignore_index=True).drop_duplicates()
probs_all.to_parquet(CACHE_FILE)

# single SN probability per oid, but keep everyone
if len(probs_all) and {"class_name","probability"}.issubset(probs_all.columns):
    sn_mask = probs_all["class_name"].astype(str).str.contains(
        r"\bSN\b|SNIa|SNII|SNIbc|SLSN", case=False, regex=True
    )
    probs_sn = (probs_all.loc[sn_mask]
                .groupby("oid", as_index=False)["probability"].max()
                .rename(columns={"probability":"alerce_sn_prob"}))
else:
    probs_sn = pd.DataFrame(columns=["oid","alerce_sn_prob"])

df = df.merge(probs_sn.rename(columns={"oid":"ZTFID"}), on="ZTFID", how="left")
df["alerce_sn_prob"] = df["alerce_sn_prob"].fillna(0.0)
df["has_alerce_probs"] = df["ZTFID"].isin(probs_sn["oid"].astype(str) if len(probs_sn) else [])

KeyboardInterrupt: 

In [141]:
from sklearn.impute import KNNImputer
from sklearn.ensemble import IsolationForest
import numpy as np
import pandas as pd

# feature columns from the cut
X = df_valid[feature_cols].copy()

#  infinities -> NaN so the imputer can handle them
#X = X.replace([np.inf, -np.inf], np.nan)

# 3) KNN-impute on the filtered rows only
knn_imp = KNNImputer(n_neighbors=5, weights="uniform")
X_imp = knn_imp.fit_transform(X)

# keep column names row index 
X_imp = pd.DataFrame(X_imp, index=X.index, columns=X.columns)



In [142]:
# IsolationForest on the imputed *filtered* features
iso = IsolationForest(
    n_estimators=300,
    contamination="auto",
    random_state=42,
    n_jobs=-1
)
iso.fit(X_imp)

# Scores & predictions
scores   = pd.Series(iso.decision_function(X_imp), index=X_imp.index)   # higher = more normal
raw_pred = pd.Series(iso.predict(X_imp),           index=X_imp.index)   # -1 anomaly, 1 normal
anomaly  = (raw_pred == -1).astype(int)                                  # 1 = anomaly

rank = scores.rank(method="first", ascending=True).astype(int)           # 1 = most anomalous

print("Estimated anomaly rate:", float(anomaly.mean()))

#  outputs using IDs from the filtered frame
out = pd.DataFrame({
    "ZTFID": df_valid["ZTFID"],
    "iso_score": scores,
    "iso_anomaly": anomaly,
    "iso_rank": rank
})

# 7) (Optional) preview with some context columns if they exist, again from df_valid
ctx_base = ["t0","r_duration_above_half_flux","g_peak_mag","r_peak_mag","mean_g-r","features_valid"]
context_cols = [c for c in ctx_base if c in df_valid.columns]

preview = pd.concat(
    [
        df_valid[["ZTFID"] + context_cols].reset_index(drop=True),
        out[["iso_score","iso_anomaly","iso_rank"]].reset_index(drop=True),
    ],
    axis=1
)

# Show 10 most anomalous
display(preview.sort_values("iso_rank").head(10))

Estimated anomaly rate: 0.03213380638621389


,ZTFID,t0,r_duration_above_half_flux,g_peak_mag,r_peak_mag,mean_g-r,features_valid,iso_score,iso_anomaly,iso_rank
37,ZTF18aabilqu,58360.513079,2208.998079,15.248654,14.825365,0.808560,True,-0.260792,1,1
383,ZTF18abtvajf,58368.229873,1561.892824,17.080242,16.883101,1.151374,True,-0.258203,1,2
19,ZTF18aabgroi,58370.468461,1922.073449,17.732500,17.965200,0.467153,False,-0.255044,1,3
2509,ZTF20acxhear,58456.247141,2073.317847,17.401100,16.331100,1.483282,True,-0.252363,1,4
586,ZTF18adisfiy,58252.413067,2062.203727,16.432600,16.021099,0.525252,True,-0.248432,1,5
160,ZTF18aaqzomm,58252.411609,2399.220775,17.181522,17.578285,-0.164648,True,-0.246901,1,6
747,ZTF19abztmpq,58833.142859,1465.955405,15.490965,13.508320,1.727393,False,-0.246695,1,7
393,ZTF18abvgkcg,58372.516979,2363.734931,18.483944,18.032764,0.610909,False,-0.234363,1,8
567,ZTF18adbihdx,58480.438935,1593.791146,17.565054,17.129118,0.497653,True,-0.231593,1,9
289,ZTF18abguzyx,58311.381088,1297.707442,17.916912,17.898600,0.156563,False,-0.230166,1,10


### ALERCE

In [127]:
# --- A: ALeRCE parsing helpers & settings ---
from alerce.search import AlerceSearch
from alerce.exceptions import APIError
import pandas as pd, numpy as np, time, random
from pathlib import Path

al = AlerceSearch()

# Which classes count as "SN-like"
SN_CLASSES = {"SNIa", "SNIbc", "SNII", "SLSN"}

def _prefer_lc_block(df):
    if "classifier_name" in df.columns:
        m = df["classifier_name"].str.contains("lc", case=False, na=False)
        if m.any():
            return df.loc[m]
    return df

def _long_form(df):
    """Return df with columns ['class','probability','classifier_name'] regardless of API format."""
    if df is None or len(df) == 0:
        return pd.DataFrame(columns=["class","probability","classifier_name"])

    # Try long form first
    cols_lower = {c.lower(): c for c in df.columns}
    class_col = next((cols_lower.get(c) for c in ["class","class_name","target_class","predicted_class","label"] if cols_lower.get(c)), None)
    prob_col  = next((cols_lower.get(c) for c in ["probability","class_probability","prob","score","ml_score"] if cols_lower.get(c)), None)

    if class_col and prob_col:
        out = df[[class_col, prob_col] + ([c for c in df.columns if c == "classifier_name"])].copy()
        out.rename(columns={class_col: "class", prob_col: "probability"}, inplace=True)
        return out

    # Wide form: class probs as columns
    class_like_cols = [c for c in df.columns if c.upper() in {s.upper() for s in SN_CLASSES} or c.isupper()]
    keep = ["classifier_name"] if "classifier_name" in df.columns else []
    if len(class_like_cols) >= 2:
        long = df[keep + class_like_cols].melt(id_vars=keep, var_name="class", value_name="probability")
        return long

    return pd.DataFrame(columns=["class","probability","classifier_name"])

def sn_score_from_pdf(pdf, prefer_lc=True):
    if pdf is None or len(pdf) == 0:
        return 0.0, None, None
    dfp = _prefer_lc_block(pdf.copy()) if prefer_lc else pdf.copy()
    lf = _long_form(dfp)
    if lf.empty: return 0.0, None, None
    lf["probability"] = pd.to_numeric(lf["probability"], errors="coerce")
    lf = lf.dropna(subset=["probability"])
    if not len(lf): return 0.0, None, None
    top_idx = lf["probability"].idxmax()
    top_class = lf.loc[top_idx, "class"]
    top_prob  = float(lf.loc[top_idx, "probability"])
    sn_sum = float(lf[lf["class"].str.upper().isin({s.upper() for s in SN_CLASSES})]["probability"].sum())
    return sn_sum, top_class, top_prob

def fetch_probs_one(oid, retries=5, base_sleep=0.8):
    for attempt in range(retries):
        try:
            return al.query_probabilities(oid=oid, format="pandas")
        except APIError:
            time.sleep(base_sleep * (2**attempt) + random.uniform(0, 0.5))
    return None


In [129]:
# --- B: Simple on-disk cache so you don't re-query the same ZTFIDs ---
CACHE_PATH = Path("alerce_prob_cache.parquet")

def load_cache():
    if CACHE_PATH.exists():
        return pd.read_parquet(CACHE_PATH)
    return pd.DataFrame(columns=["ZTFID","alerce_sn_score","alerce_top_class","alerce_top_prob"])

def update_cache(cache_df, new_rows):
    cache_df = pd.concat([cache_df, pd.DataFrame(new_rows)], ignore_index=True)
    cache_df = cache_df.drop_duplicates(subset=["ZTFID"], keep="last")
    cache_df.to_parquet(CACHE_PATH, index=False)
    return cache_df


In [133]:
# --- C: Select top-N most anomalous and fetch ALeRCE for those only ---
TOP_N = 10   # change to 10, 25, 100 as you like
THRESH = 0.5 # SN-like if summed SN prob >= 0.5

priority_ids = (
    out.sort_values("iso_rank")["ZTFID"]
      .dropna().drop_duplicates().head(TOP_N).tolist()
)

cache = load_cache()
already = set(cache["ZTFID"]) if len(cache) else set()
to_fetch = [z for z in priority_ids if z not in already]

new_rows = []
for oid in to_fetch:
    pdf = fetch_probs_one(oid)
    sn_sum, top_class, top_prob = sn_score_from_pdf(pdf, prefer_lc=True)
    new_rows.append({"ZTFID": oid,
                     "alerce_sn_score": sn_sum,
                     "alerce_top_class": top_class,
                     "alerce_top_prob": top_prob})

if new_rows:
    cache = update_cache(cache, new_rows)

# Merge cached results onto *all* rows, so non-top-N simply have NaNs for now
out_plus = out.merge(cache, on="ZTFID", how="left")
out_plus["alerce_sn_like"] = (out_plus["alerce_sn_score"] >= THRESH).astype(float).fillna(0).astype(int)

# Quick look at the top-N with labels
display(out_plus.sort_values("iso_rank").head(TOP_N))


/var/folders/bm/dwb18xdd4yg54l7ngw__81hw0000gn/T/ipykernel_41352/3353377851.py:10: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  cache_df = pd.concat([cache_df, pd.DataFrame(new_rows)], ignore_index=True)


ImportError: Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - Missing optional dependency 'pyarrow'. pyarrow is required for parquet support. Use pip or conda to install pyarrow.
 - Missing optional dependency 'fastparquet'. fastparquet is required for parquet support. Use pip or conda to install fastparquet.

In [ ]:
### No caching and top 100, does not run 

In [125]:
from alerce.search import AlerceSearch
import time, random
from alerce.exceptions import APIError

al = AlerceSearch() 

# classes count as supernovaish
SN_CLASSES = {"SNIa", "SNIbc", "SNII", "SLSN"}  

def _prefer_lc_block(df):
    if "classifier_name" in df.columns:
        m = df["classifier_name"].str.contains("lc", case=False, na=False)
        if m.any():
            return df.loc[m]
    return df

def _long_form(df):
    """Return df in long form with columns: ['class','probability','classifier_name']"""
    cols = set(c.lower() for c in df.columns)

    # Case A: already long form (class + probability exist under some names)
    class_col = None
    for cand in ["class", "class_name", "target_class", "predicted_class", "label"]:
        if cand in df.columns:
            class_col = cand
            break
        if cand in cols:  # case-insensitive
            class_col = next(c for c in df.columns if c.lower() == cand)
            break

    prob_col = None
    for cand in ["probability", "class_probability", "prob", "score", "ml_score"]:
        if cand in df.columns:
            prob_col = cand
            break
        if cand in cols:
            prob_col = next(c for c in df.columns if c.lower() == cand)
            break

    if class_col and prob_col:
        out = df[[class_col, prob_col] + [c for c in df.columns if c == "classifier_name"]].copy()
        out.rename(columns={class_col: "class", prob_col: "probability"}, inplace=True)
        return out

    # Case B: wide form (each class is a column of probabilities)
    # Detect by intersecting known class names with columns
    class_like_cols = [c for c in df.columns if c.upper() in {s.upper() for s in SN_CLASSES} or c.isupper()]
    # If we found several upper-case class columns, assume wide form
    if len(class_like_cols) >= 2:
        # Keep classifier_name if present
        keep = ["classifier_name"] if "classifier_name" in df.columns else []
        wide = df[keep + class_like_cols].copy()
        # Melt to long
        long = wide.melt(id_vars=keep, var_name="class", value_name="probability")
        return long

    # Fallback: no recognizable structure
    return pd.DataFrame(columns=["class","probability","classifier_name"])

def sn_score_from_pdf(pdf, prefer_lc=True):
    """
    Compute:
      - sn_sum: sum of SN-subclass probabilities
      - top_class: overall top class name
      - top_prob : its probability
    Works for both long and wide ALeRCE formats.
    """
    if pdf is None or len(pdf) == 0:
        return 0.0, None, None

    dfp = pdf.copy()
    if prefer_lc:
        dfp = _prefer_lc_block(dfp)

    lf = _long_form(dfp)
    if lf.empty:
        # Could not parse structure; be graceful
        return 0.0, None, None

    # Drop NaNs, coerce probability numeric
    lf = lf.copy()
    lf["probability"] = pd.to_numeric(lf["probability"], errors="coerce")
    lf = lf.dropna(subset=["probability"])

    # Top class overall
    if len(lf):
        top_idx = lf["probability"].idxmax()
        top_class = lf.loc[top_idx, "class"]
        top_prob  = float(lf.loc[top_idx, "probability"])
    else:
        top_class, top_prob = None, None

    # Sum over SN subclasses (case-insensitive)
    lf_sn = lf[lf["class"].str.upper().isin({s.upper() for s in SN_CLASSES})]
    sn_sum = float(lf_sn["probability"].sum()) if len(lf_sn) else 0.0

    return sn_sum, top_class, top_prob


# --- Apply to your Isolation Forest outputs ---
#  `out` from your previous step with column "ZTFID"
# avoid hammering the API,  deduplicate IDs first:
unique_ids = out["ZTFID"].dropna().unique().tolist()

records = []
for oid in unique_ids:
    pdf = fetch_probs_one(oid)
    sn_sum, top_class, top_prob = sn_score_from_pdf(pdf, prefer_lc=True)
    records.append({"ZTFID": oid, "alerce_sn_score": sn_sum,
                    "alerce_top_class": top_class, "alerce_top_prob": top_prob})

alerce_df = pd.DataFrame.from_records(records)

# Choose your threshold: e.g., ≥0.5 total SN probability counts as SN-like
THRESH = 0.5
alerce_df["alerce_sn_like"] = (alerce_df["alerce_sn_score"] >= THRESH).astype(int)

# Merge back onto your results
out_plus = out.merge(alerce_df, on="ZTFID", how="left")

# Prioritise and preview the most anomalous but SN-like candidates
preview_sn_like = out_plus.sort_values("iso_rank").query("alerce_sn_like == 1").head(20)
display(preview_sn_like)

# Quick sanity check: how many of your anomalies are SN-like by ALeRCE?
rate_sn_like_in_top_100 = (
    out_plus.sort_values("iso_rank").head(100)["alerce_sn_like"].mean()
)
print(f"SN-like rate in top-100 anomalies (by ALeRCE): {rate_sn_like_in_top_100:.2%}")


KeyboardInterrupt: 

### Work/Exploring

In [ ]:
# Use SN score to target objects with high likelihood of being SN
df_valid["sn_like_score"] = (
    (amp_max.clip(0, 2) / 2.0) * 0.5   # amplitude: bigger = more SN-like
    + (df_valid["g_rise_time"].between(5, 60).astype(float)) * 0.25
    + (df_valid["g_decline_time"].between(10, 150).astype(float)) * 0.25
)

In [90]:
# Recompute amp_max on df_cut to keep indices aligned, then carry into df_valid
amp_max_cut = pd.concat([df_cut["g_amplitude"], df_cut["r_amplitude"]], axis=1).max(axis=1, skipna=True)
df_cut["amp_flag"] = amp_max_cut.ge(0.5)

# Your rise/decline sanity (keep NaNs)
mask_valid = (
    df_cut["g_rise_time"].between(0, 200, inclusive="neither")
    & df_cut["g_decline_time"].between(0, 400, inclusive="neither")
)
df_valid = df_cut.loc[mask_valid | df_cut["g_rise_time"].isna()].copy()

# Build sn_like_score on df_valid (use a version of amp_max aligned to df_valid)
amp_max_valid = amp_max_cut.loc[df_valid.index]
df_valid["sn_like_score"] = (
    (amp_max_valid.clip(0, 2) / 2.0) * 0.5
    + (df_valid["g_rise_time"].between(5, 60).astype(float)) * 0.25
    + (df_valid["g_decline_time"].between(10, 150).astype(float)) * 0.25
)

In [94]:
!pip install tenacity

In [96]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type

CACHE_DIR = Path("alerce_cache"); CACHE_DIR.mkdir(exist_ok=True)
PROB_CACHE = CACHE_DIR / "probabilities.csv"

oids = df_valid["ZTFID"].astype(str).dropna().unique().tolist()

def _load_cache(path):
    return pd.read_csv(path) if path.exists() and path.stat().st_size > 0 else pd.DataFrame()

def _append_cache(path, df_new):
    if df_new is None or len(df_new) == 0: return
    if path.exists() and path.stat().st_size > 0:
        old = pd.read_csv(path)
        cat = pd.concat([old, df_new], ignore_index=True).drop_duplicates()
    else:
        cat = df_new.drop_duplicates()
    cat.to_csv(path, index=False)

@retry(stop=stop_after_attempt(3), wait=wait_exponential(multiplier=1, min=1, max=10),
       retry=retry_if_exception_type(Exception))
def _query_probs_one(oid):
    dfp = al.query_probabilities(oid=oid, format="pandas")
    if dfp is not None and len(dfp):
        dfp["oid"] = oid
    return dfp

def fetch_probabilities_batched(oids, max_workers=8):
    cached = _load_cache(PROB_CACHE)
    have = set(cached["oid"].astype(str)) if ("oid" in cached.columns and len(cached)) else set()
    todo = [o for o in oids if o not in have]
    print(f"probabilities: {len(have)} cached, {len(todo)} to fetch")

    rows = []
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futs = [ex.submit(_query_probs_one, oid) for oid in todo]
        for fut in as_completed(futs):
            try:
                res = fut.result()
                if res is not None and len(res):
                    rows.append(res)
            except Exception:
                pass

    df_new = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()
    _append_cache(PROB_CACHE, df_new)
    return _load_cache(PROB_CACHE)

# run it
prob_df_all = fetch_probabilities_batched(oids, max_workers=8)
prob_df_all.shape


probabilities: 0 cached, 20065 to fetch


KeyboardInterrupt: 

In [86]:
from alerce.core import Alerce
al = Alerce()

oids = df_valid["ZTFID"].astype(str).unique().tolist()

# Function to find probabilty of SNs from AlerCE
def fetch_probabilities(oids):
    all_probs = []
    for oid in oids:
        try:
            prob = al.query_probabilities(oid=oid, format="pandas")
            if prob is not None and len(prob):
                prob["oid"] = oid
                all_probs.append(prob)
        except Exception:
            continue
    return pd.concat(all_probs, ignore_index=True) if all_probs else pd.DataFrame()

#prob_df = fetch_probabilities(oids)
# To test:
sample_oids = oids[:50]
prob_df = fetch_probabilities(sample_oids) 


In [88]:
print(prob_df)

                          classifier_name     classifier_version class_name  \
0                           lc_classifier  hierarchical_rf_1.1.0       SNIa   
1                  lc_classifier_periodic  hierarchical_rf_1.1.0        LPV   
2                lc_classifier_stochastic  hierarchical_rf_1.1.0        QSO   
3                       lc_classifier_top  hierarchical_rf_1.1.0  Transient   
4                 lc_classifier_transient  hierarchical_rf_1.1.0       SNIa   
..                                    ...                    ...        ...   
879        lc_classifier_BHRF_forced_phot                  2.1.0      RRLab   
880  LC_classifier_ATAT_forced_phot(beta)             1.1.0_beta       RRLc   
881        lc_classifier_BHRF_forced_phot                  2.1.0       RRLc   
882  LC_classifier_ATAT_forced_phot(beta)             1.1.0_beta       DSCT   
883        lc_classifier_BHRF_forced_phot                  2.1.0       DSCT   

     probability  ranking           oid  
0        

In [ ]:
# 'ZTFID'